# Demo: Parallel Reassemble Pipeline

Fan out one large event collection across `NUM_SHARDS` parallel invocations of `roboto-to-lerobot-v3_0`, then merge the per-invocation LeRobot datasets server-side via the `lerobot-merge` action.

**Pipeline**
1. Load the parent event collection (`COLLECTION_ID`).
2. LPT bin-pack its events into `NUM_SHARDS` sub-collections balanced by event duration (mirrors `_partition_shards` inside the conversion action).
3. Invoke `roboto-to-lerobot-v3_0` once per sub-collection, in parallel. Each invocation lands on its own 16-vCPU node and still uses the action's internal `shard_count=3`.
4. Wait for every conversion invocation to reach a terminal status.
5. Invoke `lerobot-merge` against the per-invocation outputs. The merge runs on Roboto compute (`lerobot.datasets.aggregate.aggregate_datasets`): videos are stream-copy concatenated (no re-encode), parquet columns are reindexed in one pass, tasks are unioned, stats aggregated.
6. Wait for the merge to complete and read its manifest.

**Why server-side merge**: keeps the video bytes inside AWS. A laptop-side merge would download every shard's video (potentially many GB) over the WAN, run the same merge locally, and re-upload. The hosted action does the same work AWS-internal — same merge cost, no WAN round-trip.

## Initialization

Set `COLLECTION_ID` (the parent event collection you want to convert in parallel). `NUM_SHARDS` controls the fan-out width; 4 is a reasonable default.

In [ ]:
# Inputs ------------------------------------------------------------------
COLLECTION_ID = ""  # parent event collection to fan out across NUM_SHARDS invocations
NUM_SHARDS = 4

# Conversion-action wiring (same as the single-invocation demo) ----------
CONTRACTS_DATASET_ID = "ds_xxxxxxxxxxxx"  # your dataset that holds the contract YAML
CONTRACT = "contract_demo_reassemble.yaml"
CONVERT_ACTION_NAME = "roboto-to-lerobot-v3_0"

# Server-side merge action ------------------------------------------------
MERGE_ACTION_NAME = "lerobot-merge"

# Per-invocation timeouts (seconds) for waiting on terminal status
CONVERT_TIMEOUT_S = 2 * 60 * 60
MERGE_TIMEOUT_S = 1 * 60 * 60

assert COLLECTION_ID, "Set COLLECTION_ID to the parent event collection ID before running."
assert NUM_SHARDS >= 1, "NUM_SHARDS must be >= 1."

In [ ]:
import datetime
import json
import pathlib
import tempfile

import roboto
from notebook_helpers import partition_events_lpt, tail_until_done, wait_all_invocations
from roboto.domain import actions

roboto_client = roboto.RobotoClient.from_env()

## Load the parent collection and hydrate its events

`Collection.from_id` with the default `content_mode=Full` hydrates every event resource in one round-trip — we get `Event` objects with `start_time` and `end_time` that drive the LPT partition.

In [ ]:
parent_collection = roboto.Collection.from_id(COLLECTION_ID)
assert parent_collection.record.resource_type == roboto.CollectionResourceType.Event, (
    f"Collection {COLLECTION_ID} must have resource_type=Event "
    f"(got {parent_collection.record.resource_type})."
)

# `Collection.from_id` defaults to content_mode=Full, which hydrates every event
# resource server-side. We materialize them in-process instead of looping
# `Event.from_id` (saving N round trips) — same pattern the action uses.
hydrated = parent_collection.record.resources.get(roboto.CollectionResourceType.Event, [])
events = [
    roboto.Event(roboto.EventRecord.model_validate(r) if isinstance(r, dict) else r)
    for r in hydrated
]

missing = parent_collection.record.missing.get(roboto.CollectionResourceType.Event, [])
if missing:
    print(
        f"  WARNING: {len(missing)} event(s) referenced by the collection could not be "
        f"hydrated (deleted or inaccessible) and will be skipped: "
        f"{sorted(ref.resource_id for ref in missing)}"
    )

print(
    f"Parent collection {COLLECTION_ID} (version {parent_collection.record.version}): "
    f"{len(events)} hydrated event(s)"
)
assert events, "Parent collection has no hydratable events."

## Partition events into `NUM_SHARDS` sub-collections (LPT bin-pack)

Sort events heaviest-first by `end_time - start_time` (frame-count proxy when fps is constant), then greedy-assign each to the sub-shard with the smallest running total. Each sub-shard gets a fresh `Collection` whose ID feeds one parallel invocation.

In [ ]:
shards = partition_events_lpt(events, NUM_SHARDS)
for i, shard in enumerate(shards):
    total_ns = sum(int(e.end_time) - int(e.start_time) for e in shard)
    print(f"  shard {i:02d}: {len(shard):4d} events, total span {total_ns/1e9:9.2f} s")

batch_ts = datetime.datetime.now(datetime.UTC).isoformat(timespec="seconds")
sub_collections = []
for i, shard in enumerate(shards):
    if not shard:
        # Possible when NUM_SHARDS > len(events); skip empty shards.
        sub_collections.append(None)
        continue
    sub = roboto.Collection.create(
        name=f"parallel-{COLLECTION_ID}-{i:02d}-{batch_ts}",
        description=(
            f"LPT shard {i+1}/{NUM_SHARDS} of parent collection {COLLECTION_ID}. "
            f"Used by the parallel reassemble notebook."
        ),
        resource_type=roboto.CollectionResourceType.Event,
        event_ids=[e.event_id for e in shard],
    )
    sub_collections.append(sub)
    print(f"  created shard {i:02d} collection {sub.collection_id} with {len(shard)} event(s)")

## Create the per-invocation output dataset

All `NUM_SHARDS` invocations upload to this one dataset under their own `<invocation_id>/combined/` prefix. The final merged dataset is created at the end of the notebook as a separate Roboto dataset.

In [ ]:
invocation_outputs_dataset = roboto.Dataset.create(
    name=f"parallel-reassemble-shards-{COLLECTION_ID}-{batch_ts}",
    description=(
        f"Per-invocation LeRobot outputs for parallel reassemble of parent "
        f"collection {COLLECTION_ID} ({NUM_SHARDS}-way fan-out)."
    ),
    tags=["lerobot", "parallel-reassemble"],
    metadata={
        "source_collection_id": COLLECTION_ID,
        "source_collection_version": parent_collection.record.version,
    },
)
print(f"Per-invocation outputs will land in {invocation_outputs_dataset.dataset_id}")

## Fan out: invoke the conversion action once per sub-collection

Each invocation runs on its own 16-vCPU node and uses the action's internal `shard_count` (default 3) for per-node parallelism. The notebook does no log tailing here — `wait_all_invocations` overlaps the polling across threads so the wall-clock is bounded by the slowest invocation, not the sum.

In [ ]:
convert_action = actions.Action.from_name(CONVERT_ACTION_NAME)

invocations = []
for i, sub in enumerate(sub_collections):
    if sub is None:
        continue
    iv = convert_action.invoke(
        invocation_source=actions.InvocationSource.Manual,
        data_source_id=CONTRACTS_DATASET_ID,
        input_data=[CONTRACT],
        upload_destination=actions.InvocationUploadDestination.dataset(
            invocation_outputs_dataset.dataset_id
        ),
        parameter_values={
            "collection_id": sub.collection_id,
            "contract": CONTRACT,
        },
    )
    invocations.append(iv)
    print(f"  shard {i:02d}: invocation {iv.id} (collection {sub.collection_id})")

print(f"\nWaiting for {len(invocations)} conversion invocation(s) to reach terminal status (timeout {CONVERT_TIMEOUT_S}s)...")
statuses = wait_all_invocations(invocations, timeout=CONVERT_TIMEOUT_S, poll_interval=10)
for iv, status in zip(invocations, statuses, strict=True):
    print(f"  {iv.id}: {status}")

failed = [iv for iv, s in zip(invocations, statuses, strict=True) if s != actions.InvocationStatus.Completed]
assert not failed, f"{len(failed)} conversion invocation(s) did not complete: {[iv.id for iv in failed]}"

## Create the merged output dataset

The `lerobot-merge` action will write its merged LeRobot dataset into a fresh Roboto dataset. Tag with the parent collection ID and per-shard invocation IDs so the provenance chain — parent collection → `NUM_SHARDS` sub-collections → conversion invocations → merge invocation → merged dataset — is traceable from metadata alone.

In [ ]:
merged_dataset = roboto.Dataset.create(
    name=f"parallel-reassemble-merged-{COLLECTION_ID}-{batch_ts}",
    description=(
        f"Merged LeRobot dataset from {NUM_SHARDS}-way parallel reassemble of parent "
        f"collection {COLLECTION_ID}. Conversion invocations: {[iv.id for iv in invocations]}"
    ),
    tags=["lerobot", "parallel-reassemble-merged"],
    metadata={
        "source_collection_id": COLLECTION_ID,
        "source_collection_version": parent_collection.record.version,
        "conversion_invocation_ids": [iv.id for iv in invocations],
    },
)
print(f"Merged dataset will land in {merged_dataset.dataset_id}")

## Invoke `lerobot-merge` on Roboto compute

The action downloads each `<conversion_invocation_id>/combined/` from `invocation_outputs_dataset`, runs `lerobot.datasets.aggregate.aggregate_datasets`, and writes the merged dataset to `merged_dataset` under a `combined/` prefix (alongside a `manifest.json` recording source-shard provenance and merge timings). All data movement is AWS-internal.

In [ ]:
merge_action = actions.Action.from_name(MERGE_ACTION_NAME)

merge_iv = merge_action.invoke(
    invocation_source=actions.InvocationSource.Manual,
    # The shards dataset is the merge action's data source: binding it
    # lets the action read it via context.dataset and stamp its
    # metadata.invocations entry on it. input_data tells the runtime
    # which files to pre-download into context.input_dir — one "<iv>/**"
    # per conversion invocation covers each shard's combined/ tree plus
    # its sibling manifest.json and contract.yaml.
    data_source_id=invocation_outputs_dataset.dataset_id,
    input_data=[f"{iv.id}/**" for iv in invocations],
    upload_destination=actions.InvocationUploadDestination.dataset(merged_dataset.dataset_id),
    parameter_values={
        # Ordered shard list — aggregate_datasets stacks shards in this
        # order and the merged manifest's episode_to_event indices are
        # offset accordingly.
        "shard_invocation_ids": json.dumps([iv.id for iv in invocations]),
        # Stamp parent collection id/version onto the merged manifest so the
        # merged dataset's provenance matches what a single-shot conversion
        # would produce. This makes it safe to delete the sub-collections
        # and the invocation_outputs_dataset afterwards.
        "parent_collection_id": COLLECTION_ID,
        "parent_collection_version": str(parent_collection.record.version),
    },
)
print(f"Merge invocation: {merge_iv.id}")

# Stream logs while the merge runs — single invocation, so tail_until_done is fine here.
merge_status = tail_until_done(merge_iv)
assert merge_status == actions.InvocationStatus.Completed, (
    f"lerobot-merge invocation {merge_iv.id} did not complete: {merge_status}"
)


## Inspect the merged manifest

The merge action writes a self-contained `manifest.json` next to `combined/` in the merged dataset. Its shape matches what a single-shot conversion would produce — `collection_id` is the parent collection, `episode_to_event` carries globally renumbered episode indices, and dedup/skipped event records are unioned across shards. Pull just that one small file to confirm.

In [ ]:
manifest_dir = pathlib.Path(tempfile.mkdtemp(prefix="merge-manifest-"))
manifest_remote = f"{merge_iv.id}/manifest.json"
merged_dataset.download_files(
    manifest_dir,
    include_patterns=[manifest_remote],
)
manifest_path = manifest_dir / manifest_remote
assert manifest_path.is_file(), (
    f"{manifest_remote} not found under {manifest_dir} after downloading from {merged_dataset.dataset_id}"
)

merge_manifest = json.loads(manifest_path.read_text())

# Print the top-level summary fields; episode_to_event can be long so collapse it.
preview = {k: v for k, v in merge_manifest.items() if k != "episode_to_event"}
preview["episode_to_event_count"] = len(merge_manifest.get("episode_to_event") or [])
print(json.dumps(preview, indent=2))


## Sanity-check the merged totals

Cross-check the merged dataset's totals against the parent collection. A mismatch in `total_episodes` typically means some events were skipped by the conversion action (no matching topics) — each per-shard conversion `manifest.json` in `invocation_outputs_dataset` records which ones and why.

In [ ]:
expected_total_episodes = sum(len(s) for s in shards)

print(f"Parent collection:     {merge_manifest['collection_id']} (version {merge_manifest['collection_version']})")
print(f"Conversion outputs:    {invocation_outputs_dataset.dataset_id}")
print(f"Merged dataset:        {merged_dataset.dataset_id}")
print(f"Merge invocation:      {merge_iv.id}")
print(f"Contract:              {merge_manifest['contract'].get('name')} "
      f"v{merge_manifest['contract'].get('version')} "
      f"(sha256={(merge_manifest['contract'].get('sha256') or '')[:12]}...)")
print()
print(f"total_episodes:        {merge_manifest['total_episodes']} (expected ~{expected_total_episodes})")
print(f"total_frames:          {merge_manifest['total_frames']}")
print(f"total_tasks:           {merge_manifest['total_tasks']}")
print(f"fps:                   {merge_manifest['fps']}")
print(f"episode_to_event rows: {len(merge_manifest.get('episode_to_event') or [])}")
print(f"skipped_events:        {merge_manifest['skipped_events_summary']['count']}")
print(f"dedup duplicate groups:{merge_manifest['dedup_summary']['duplicate_groups']}")

if merge_manifest["total_episodes"] != expected_total_episodes:
    print(
        f"\nNOTE: merged total_episodes ({merge_manifest['total_episodes']}) differs "
        f"from len(parent events) ({expected_total_episodes}). Some events were probably "
        f"skipped by the conversion action — see merge_manifest['skipped_events'] for "
        f"per-event reasons (already unioned across shards)."
    )

print(
    "\nThe merged manifest is self-contained: parent collection, contract, per-event "
    "mapping, dedup, and skipped events are all inlined. The sub-collections "
    f"({[c.collection_id for c in sub_collections if c is not None]}) "
    f"and the intermediate outputs dataset ({invocation_outputs_dataset.dataset_id}) "
    "can now be deleted without losing traceability."
)